# Intelligent Customer Segmentation & Behavioral Analytics
## Notebook 03: Clustering Algorithms, Evaluation & PCA Profiling

### Overview
In this notebook, we implement, compare, and evaluate multiple unsupervised clustering approaches:
1. **K-Means Clustering** (with Elbow Method & Silhouette optimization)
2. **DBSCAN** (Density-Based Spatial Clustering for noise/outlier discovery)
3. **Agglomerative Hierarchical Clustering** (with Dendrogram representation)
4. **Cluster Evaluation Metrics** (Silhouette, Davies-Bouldin, Calinski-Harabasz)
5. **Dimensionality Reduction via PCA** (2D/3D visualization)
6. **Cluster Profiling & Business Interpretation**

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.decomposition import PCA

sys.path.append(os.path.join('..', 'training'))
from preprocessing import engineer_features, CLUSTERING_FEATURES, fit_scaler, fit_pca
from evaluate_clusters import compute_clustering_metrics, compute_elbow_and_silhouette_range, benchmark_clustering_algorithms
from cluster_profiling import generate_cluster_profiles

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### 1. Load Data & Prepare Scaled Features

In [ ]:
data_path = os.path.join('..', 'data', 'customer_data.csv')
df = pd.read_csv(data_path)
featured_df = engineer_features(df)
X = featured_df[CLUSTERING_FEATURES]

scaler = fit_scaler(X)
X_scaled = scaler.transform(X)
print(f"Feature matrix scaled: shape {X_scaled.shape}")

### 2. Algorithm 1: K-Means Clustering
#### Optimal K Determination: Elbow Curve & Silhouette Analysis

In [ ]:
curves = compute_elbow_and_silhouette_range(X_scaled, k_min=2, k_max=8)
k_vals = curves['k_values']
inertias = curves['inertias']
silhouettes = curves['silhouettes']

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Elbow Curve
axes[0].plot(k_vals, inertias, marker='o', color='#1D3557', linewidth=2)
axes[0].set_title('Elbow Method (Inertia / WCSS vs K)')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia')

# Silhouette Curve
axes[1].plot(k_vals, silhouettes, marker='s', color='#E63946', linewidth=2)
axes[1].set_title('Silhouette Score vs K')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')

plt.tight_layout()
plt.show()

#### Train Optimal K-Means Model ($K=5$)

In [ ]:
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=15)
km_labels = kmeans.fit_predict(X_scaled)
featured_df['KMeans_Cluster'] = km_labels

km_metrics = compute_clustering_metrics(X_scaled, km_labels)
print("K-Means Metrics:", km_metrics)

### 3. Algorithm 2: DBSCAN (Density-Based Spatial Clustering)
Identifies dense clusters and flags noise/outlier customers (label = -1).

In [ ]:
dbscan = DBSCAN(eps=1.2, min_samples=15)
db_labels = dbscan.fit_predict(X_scaled)

n_noise = (db_labels == -1).sum()
n_db_clusters = len(set(db_labels)) - (1 if -1 in db_labels else 0)
print(f"DBSCAN found {n_db_clusters} clusters and {n_noise} noise points ({n_noise/len(X_scaled)*100:.2f}% of data).")

db_metrics = compute_clustering_metrics(X_scaled, db_labels)
print("DBSCAN Metrics:", db_metrics)

### 4. Algorithm 3: Hierarchical / Agglomerative Clustering & Dendrogram

In [ ]:
# Sample for dendrogram clarity
sample_idx = np.random.choice(len(X_scaled), size=120, replace=False)
linked = linkage(X_scaled[sample_idx], method='ward')

plt.figure(figsize=(14, 6))
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=True)
plt.title('Hierarchical Dendrogram (Ward Linkage, Sample of 120 Customers)')
plt.xlabel('Customer Sample Index')
plt.ylabel('Euclidean Distance')
plt.show()

### 5. Algorithm Comparison Matrix
Comparing K-Means, DBSCAN, and Agglomerative Clustering across:
- **Silhouette Score** (Higher = better separation)
- **Davies-Bouldin Index** (Lower = tighter, better-separated clusters)
- **Calinski-Harabasz Score** (Higher = well-defined clusters)

In [ ]:
benchmark_df = benchmark_clustering_algorithms(X_scaled, k_optimal=optimal_k)
benchmark_df

### 6. Dimensionality Reduction via PCA (2D & 3D Visualization)

In [ ]:
pca = fit_pca(X_scaled, n_components=2)
coords_2d = pca.transform(X_scaled)
featured_df['PCA_1'] = coords_2d[:, 0]
featured_df['PCA_2'] = coords_2d[:, 1]

print(f"Explained Variance Ratio (2 components): {pca.explained_variance_ratio_.sum()*100:.2f}%")

plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='PCA_1', y='PCA_2',
    hue='KMeans_Cluster',
    palette='Set1',
    data=featured_df,
    alpha=0.7,
    s=40
)
plt.title('2D PCA Projection of Customer Segments')
plt.xlabel(f'PCA Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}% Variance)')
plt.ylabel(f'PCA Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}% Variance)')
plt.legend(title='Cluster ID')
plt.show()

### 7. Cluster Profiling & Business Segmentation

In [ ]:
profiles = generate_cluster_profiles(featured_df, cluster_col='KMeans_Cluster')
for cid, p in profiles.items():
    print(f"\n=========================================")
    print(f"Cluster {cid}: {p['segment_name']} ({p['tier']} Tier)")
    print(f"Customer Count: {p['customer_count']} ({p['customer_percentage']}%)")
    print(f"Avg Spend: ₹{p['metrics']['avg_spend_inr']:,.0f} | Avg Orders: {p['metrics']['avg_orders']}")
    print(f"Persona: {p['persona']}")
    print(f"Strategies:")
    for s in p['strategies']:
        print(f" - {s}")